# Path 3: Bulk product catalog + order-item seeding

Simulates several customers, each placing one or more orders, with many
catalog products sold as order-items scattered across those orders.

Covers bulk `POST /customers`, `POST /orders`, `POST /products`, `POST /order-items`, then
verification via `GET /orders/:id/items` and `GET /orders/pending-totals`.

Run top-to-bottom (e.g. `jupyter nbconvert --to notebook --execute 03_bulk_products_seed.ipynb`).

In [1]:
import os
import random
from collections import defaultdict

import requests
from faker import Faker

BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:3002/api")
fake = Faker()

CUSTOMER_COUNT = int(os.environ.get("CUSTOMER_COUNT", "200"))
ORDERS_PER_CUSTOMER = int(os.environ.get("ORDERS_PER_CUSTOMER", "20"))
TOTAL_PRODUCTS = int(os.environ.get("TOTAL_PRODUCTS", "3000"))

print(
    f"BASE_URL={BASE_URL} CUSTOMER_COUNT={CUSTOMER_COUNT} "
    f"ORDERS_PER_CUSTOMER={ORDERS_PER_CUSTOMER} TOTAL_PRODUCTS={TOTAL_PRODUCTS}"
)

BASE_URL=http://api-service:5000/api CUSTOMER_COUNT=200 ORDERS_PER_CUSTOMER=20 TOTAL_PRODUCTS=3000


## Step 1 - Bulk sign up customers with Faker

In [2]:
customers = []
for _ in range(CUSTOMER_COUNT):
    resp = requests.post(
        f"{BASE_URL}/customers",
        json={"email": fake.unique.email(), "password": fake.password(length=14)},
    )
    assert resp.status_code == 201, resp.text
    customers.append(resp.json())

print(f"Created {len(customers)} customers")

Created 200 customers


## Step 2 - Each customer places one or more orders (Faker-chosen status)

In [3]:
ORDER_STATUSES = ["pending", "shipped", "delivered"]

orders = []
for customer in customers:
    for _ in range(ORDERS_PER_CUSTOMER):
        status = random.choice(ORDER_STATUSES)
        resp = requests.post(
            f"{BASE_URL}/orders",
            json={"customer_id": customer["id"], "status": status},
        )
        assert resp.status_code == 201, resp.text
        orders.append(resp.json())

print(f"Created {len(orders)} orders across {len(customers)} customers")

Created 4000 orders across 200 customers


## Step 3 - Bulk-create Faker catalog products, then add them as line items across orders

In [4]:
def fake_product():
    return {
        "sku": f"SKU-{fake.unique.bothify(text='???-####').upper()}",
        "name": fake.unique.catch_phrase(),
        "unit_price_cents": fake.random_int(min=99, max=14999),
    }


created_products = []
created_items = []
items_by_order = defaultdict(list)

for _ in range(TOTAL_PRODUCTS):
    resp = requests.post(f"{BASE_URL}/products", json=fake_product())
    assert resp.status_code == 201, resp.text
    product = resp.json()
    created_products.append(product)

    order = random.choice(orders)
    resp = requests.post(
        f"{BASE_URL}/order-items",
        json={
            "order_id": order["id"],
            "product_id": product["id"],
            "quantity": fake.random_int(min=1, max=6),
            "unit_price_cents": product["unit_price_cents"],
        },
    )
    assert resp.status_code == 201, resp.text
    item = resp.json()
    created_items.append(item)
    items_by_order[order["id"]].append(item)

print(
    f"Created {len(created_products)} catalog products and {len(created_items)} "
    f"line items across {len(items_by_order)} distinct orders"
)

Created 3000 catalog products and 3000 line items across 2137 distinct orders


## Step 4 - Verify per-order item counts via `/orders/:id/items`

In [5]:
for order_id, expected_items in items_by_order.items():
    resp = requests.get(f"{BASE_URL}/orders/{order_id}/items")
    assert resp.status_code == 200, resp.text
    actual_items = resp.json()
    assert len(actual_items) == len(expected_items), (
        f"order {order_id}: expected {len(expected_items)} items, got {len(actual_items)}"
    )

print("All per-order item counts verified")

All per-order item counts verified


## Step 5 - Cross-check `/orders/pending-totals` for the orders left in `pending` status

In [6]:
resp = requests.get(f"{BASE_URL}/orders/pending-totals")
assert resp.status_code == 200, resp.text
pending_totals = {row["order_id"]: row for row in resp.json()}

pending_orders = [o for o in orders if o["status"] == "pending" and o["id"] in items_by_order]
checked = 0
for order in pending_orders:
    expected_total = sum(i["quantity"] * i["unit_price_cents"] for i in items_by_order[order["id"]])
    row = pending_totals.get(order["id"])
    assert row is not None, f"pending order {order['id']} missing from pending-totals"
    assert int(row["total_cents"]) == expected_total, (
        f"order {order['id']}: expected total {expected_total}, got {row['total_cents']}"
    )
    checked += 1

print(f"Verified totals for {checked} pending orders with products")
print(
    f"Bulk seed summary: {len(customers)} customers, {len(orders)} orders, "
    f"{len(created_products)} catalog products, {len(created_items)} line items"
)

AssertionError: pending order 16 missing from pending-totals